# Practice guide 03_01: classic anomaly detectors on network connections

**Goal.** Build the statistical core of an anomaly detector yourself (z-score, Mahalanobis distance, a threshold set
on normal data, average precision), compare it with the scikit-learn twins, run the classic detectors on KDD99, and
measure why a classifier is not enough for new attacks. Every `TODO` cell must be completed; the `check` cells tell
you whether you got it right.

Read `guide_03_01.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.covariance import EmpiricalCovariance, LedoitWolf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

LIB = next(p for p in (Path("../../notebooks/02-anomaly"), Path("../../../notebooks/02-anomaly")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import anomlib as an

an.table_config()

## Part A. Example 1 and KDD99 (given)

Example 1: 500 sessions of one host, $x_1$ = log bytes sent, $x_2$ = log bytes received (standardised, correlation
0.9). Point A $=(2,2)$, point B $=(1,-1)$. KDD99: deduplicated, normal rows split 60/20/20, the test set holds 100
attacks per category (u2r: 26), 1.8% of the rows.

In [ ]:
X1 = an.example1(500, seed=0)
A, B = an.EX1_POINTS["A"], an.EX1_POINTS["B"]
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
cats = sp.test["category"]
print("Example 1:", X1.shape, " KDD99 train/val/test:", Xtr.shape, Xva.shape, Xte.shape, " attacks in test:", int(y.sum()))

## Part B. z-score and Mahalanobis distance

Implement
* `zmax(X, mu, sd)`: for each row, $\max_j |x_j-\mu_j|/\sigma_j$;
* `mahalanobis2(X, mu, cov)`: for each row, $(\mathbf{x}-\boldsymbol\mu)^\top C^{-1}(\mathbf{x}-\boldsymbol\mu)$.

Use `np.linalg.inv` (or better, `np.linalg.solve`) and `np.einsum("ij,jk,ik->i", D, P, D)` to avoid a loop.

In [ ]:
def zmax(X, mu, sd):
    raise NotImplementedError


def mahalanobis2(X, mu, cov):
    raise NotImplementedError

In [ ]:
# check: the numbers of the lecture (true parameters) and the scikit-learn twin (estimated parameters)
P2 = np.vstack([A, B])
assert np.allclose(zmax(P2, an.EX1_MU, np.ones(2)), [2, 1])
assert np.allclose(mahalanobis2(P2, an.EX1_MU, an.EX1_COV), [4.2105, 20.0], atol=1e-3)
emp = EmpiricalCovariance().fit(X1)
mine = mahalanobis2(P2, X1.mean(0), np.cov(X1.T, bias=True))
print("mine:", mine.round(3), " scikit-learn:", emp.mahalanobis(P2).round(3))
assert np.allclose(mine, emp.mahalanobis(P2))
print("z-score and Mahalanobis OK")

**Question B1.** Which point does each score flag, and why? Draw (on paper) the contour $\max|z|=2$ and the
ellipse $d_M=3$ over the cloud of Example 1.

## Part C. Threshold and average precision

* `threshold(s_val, fpr)`: the $(1-\text{fpr})$ quantile of the scores of **normal** validation rows.
* `average_precision(y, s)`: sort by score (highest first); at the rank of each attack compute the precision of the
  top-ranked rows; return the mean of those precisions. It must match `average_precision_score`.
* `precision_at_base_rate(tpr, fpr, b)`: Bayes' rule of the lecture.

In [ ]:
def threshold(s_val, fpr=0.01):
    raise NotImplementedError


def average_precision(y, s):
    raise NotImplementedError


def precision_at_base_rate(tpr, fpr, b):
    raise NotImplementedError

In [ ]:
# check
rng = np.random.default_rng(0)
yy, ss = (rng.random(2000) < 0.05).astype(int), rng.random(2000)
ss[yy == 1] += 0.3
assert abs(average_precision(yy, ss) - average_precision_score(yy, ss)) < 1e-9
assert abs(threshold(np.arange(101.0), 0.01) - 99.0) < 1e-9
assert abs(precision_at_base_rate(0.95, 0.01, 1e-4) - 0.0094) < 1e-4
print("threshold, AP and base rate OK")

**Question C1.** Fill the table for TPR = 0.95. How many of 100 alarms are real at $b$ = 1/10,000 and FPR = 1%?

In [ ]:
print(pl.DataFrame([{"base rate": f"1/{round(1 / b):,}", **{f"FPR {f}": precision_at_base_rate(0.95, f, b) for f in [0.01, 0.001]}}
                    for b in [1e-2, 1e-3, 1e-4]]))  # fmt: skip

## Part D. Mahalanobis on KDD99, and the classic zoo

With 71 features (many one-hot columns) the sample covariance is singular. **Shrink** it:
$\hat C=(1-\lambda)S+\lambda\,\frac{\text{tr}(S)}{d}I$. Implement `fit_shrunk(X, lam)` returning
$(\boldsymbol\mu,\hat C)$; then compare $\lambda=0.01, 0.1$ with scikit-learn's `LedoitWolf` (which picks
$\lambda$ from the data).

In [ ]:
def fit_shrunk(X, lam):
    raise NotImplementedError

In [ ]:
# check and comparison with the twin (given)
rows = []
for lam in [0.01, 0.1]:
    mu, cov = fit_shrunk(Xtr, lam)
    rows.append({"model": f"mine, lambda={lam}", **an.evaluate(mahalanobis2(Xte, mu, cov), y, mahalanobis2(Xva, mu, cov))})
lw = LedoitWolf().fit(Xtr)
rows.append({"model": f"LedoitWolf, lambda={lw.shrinkage_:.4f}", **an.evaluate(lw.mahalanobis(Xte), y, lw.mahalanobis(Xva))})
print(pl.DataFrame(rows))

Now the zoo of the lecture (given): ten detectors, one protocol. Copy the AUPRC and the recall per category into
your notes.

In [ ]:
rows = []
for det in an.zoo(seed=0):
    det.fit(Xtr)
    s_va, s_te = det.score(Xva), det.score(Xte)
    rows.append({"detector": det.name, **an.evaluate(s_te, y, s_va), **an.recall_by_category(s_te, cats, s_va)})
print(pl.DataFrame(rows).sort("AUPRC", descending=True))

**Question D1.** Which category is hard for every detector, and why? Which detector misses most DoS rows, and why
does a flood fool it? **D2.** How does the shrinkage $\lambda$ change the result, and what does `LedoitWolf` choose?

## Part E. The Isolation Forest score

Implement `c(n)` $=2H(n-1)-2(n-1)/n$ with $H(i)=\ln i+0.5772156649$, and `iforest_score(mean_depth, n)`
$=2^{-\text{mean\_depth}/c(n)}$.

In [ ]:
def c(n):
    raise NotImplementedError


def iforest_score(mean_depth, n=256):
    raise NotImplementedError

In [ ]:
# check
assert abs(c(256) - 10.2448) < 1e-3
assert abs(iforest_score(c(256)) - 0.5) < 1e-12
print("c(256) =", round(c(256), 3), " score after 4 cuts:", round(float(iforest_score(4)), 3))

**Question E1.** What score does a point get when it is isolated after 2 cuts? After 20? Why do the trees use small
subsamples (256 rows)?

## Part F. A classifier or a one-class detector?

Implement `train_without(category)`: a random forest (200 trees, `random_state=0`, `n_jobs=-1`) trained on the
normal training rows (label 0) and the attack pool **without** `category` (label 1). Return the recall on the
test attacks of that category at the 1% validation threshold (use `an.recall_by_category`).

In [ ]:
Xp, cp = sp.X("pool"), sp.pool["category"].to_numpy()


def train_without(category):
    raise NotImplementedError

In [ ]:
# driver (given)
pca = an.PCARecon().fit(Xtr)
one_class = an.recall_by_category(pca.score(Xte), cats, pca.score(Xva))
print(pl.DataFrame([{"unseen category": cat, "random forest": train_without(cat), "PCA (one-class)": one_class[cat]}
                    for cat in an.CATEGORIES]))  # fmt: skip

**Question F1.** For which unseen category does the forest fail, and why? What does that say about deploying only a
classifier? **F2.** Propose a combination of the two kinds of detector for a real network.